In [17]:
import json
import os

import requests
from dotenv import load_dotenv

In [18]:
load_dotenv()  # Load environment variables from .env file
max_batch_size = int(os.getenv("MAX_BATCH_INGEST_SIZE", 10))

In [19]:
wiki_link_array = ["https://en.wikipedia.org/wiki/Mathematics",
                   "https://en.wikipedia.org/wiki/Number_theory",
                   "https://en.wikipedia.org/wiki/Calculus",
                   "https://en.wikipedia.org/wiki/Linear_algebra",
                   "https://en.wikipedia.org/wiki/Euclidean_geometry",
                   "https://en.wikipedia.org/wiki/Topology",
				   "https://en.wikipedia.org/wiki/Probability_theory",
				   "https://en.wikipedia.org/wiki/Statistics",
				   "https://en.wikipedia.org/wiki/Combinatorics",
				   "https://en.wikipedia.org/wiki/Graph_theory",
				   "https://en.wikipedia.org/wiki/Set_theory",
				   "https://en.wikipedia.org/wiki/Logic",
				   "https://en.wikipedia.org/wiki/Number_system",
				   "https://en.wikipedia.org/wiki/Algebraic_geometry",
				   "https://en.wikipedia.org/wiki/Differential_equations",
				   "https://en.wikipedia.org/wiki/Mathematical_analysis",
				   "https://en.wikipedia.org/wiki/Functional_analysis",
				   "https://en.wikipedia.org/wiki/Complex_analysis",
				   "https://en.wikipedia.org/wiki/Real_analysis",
				   "https://en.wikipedia.org/wiki/Numerical_analysis"]

In [20]:
print("Number of links to process: ", len(wiki_link_array))

Number of links to process:  20


In [21]:
# Split the list into sublists of at most 10 strings
batches = [wiki_link_array[i : i + max_batch_size] for i in range(0, len(wiki_link_array), max_batch_size)]

# Print the result
for idx, batch in enumerate(batches):
    print(f"Batch {idx + 1} (Size {len(batch)}): {batch}")

Batch 1 (Size 10): ['https://en.wikipedia.org/wiki/Mathematics', 'https://en.wikipedia.org/wiki/Number_theory', 'https://en.wikipedia.org/wiki/Calculus', 'https://en.wikipedia.org/wiki/Linear_algebra', 'https://en.wikipedia.org/wiki/Euclidean_geometry', 'https://en.wikipedia.org/wiki/Topology', 'https://en.wikipedia.org/wiki/Probability_theory', 'https://en.wikipedia.org/wiki/Statistics', 'https://en.wikipedia.org/wiki/Combinatorics', 'https://en.wikipedia.org/wiki/Graph_theory']
Batch 2 (Size 10): ['https://en.wikipedia.org/wiki/Set_theory', 'https://en.wikipedia.org/wiki/Logic', 'https://en.wikipedia.org/wiki/Number_system', 'https://en.wikipedia.org/wiki/Algebraic_geometry', 'https://en.wikipedia.org/wiki/Differential_equations', 'https://en.wikipedia.org/wiki/Mathematical_analysis', 'https://en.wikipedia.org/wiki/Functional_analysis', 'https://en.wikipedia.org/wiki/Complex_analysis', 'https://en.wikipedia.org/wiki/Real_analysis', 'https://en.wikipedia.org/wiki/Numerical_analysis']


In [24]:
job_ids = []
for idx, batch in enumerate(batches):
	url_array = []
	for url in batch:
		elements = url.split("/")
		title = elements[-1]
		url_array.append({"url": url, "title": title})
	payload = {"documents": url_array}
	response = requests.post("http://localhost:8000/api/v1/documents/ingest", json=payload)
	if response.status_code == 202:
		job_id = response.json().get("main_job_id")
		job_ids.append(job_id)
		print(f"Batch {idx + 1} submitted successfully. Job ID: {job_id}")
	else:
		print(f"Failed to submit batch {idx + 1}. Status code: {response.status_code}, Response: {response.text}")

print("All batches submitted. Job IDs:", job_ids)

Batch 1 submitted successfully. Job ID: 4bb9ece8-916f-4c4b-9526-16b7e7cff583
Batch 2 submitted successfully. Job ID: 7979ae9e-c2e0-4aa0-b809-854803d6deb5
All batches submitted. Job IDs: ['4bb9ece8-916f-4c4b-9526-16b7e7cff583', '7979ae9e-c2e0-4aa0-b809-854803d6deb5']


In [ ]:
for job_id in job_ids:
	response = requests.get(f"http://localhost:8000/api/v1/documents/status/{job_id}")
	if response.status_code == 200:
		response_json = response.json()
		status = response_json["status"]
		print(f"Job ID: {job_id}, Status: {status}")
		print(f"Response received: {json.dumps(response_json, indent=4)}")
	else:
		print(f"Failed to get status for Job ID: {job_id}. Status code: {response.status_code}, Response: {response.text}")

Job ID: 4bb9ece8-916f-4c4b-9526-16b7e7cff583, Status: PROCESSING
Response received: {'main_job_id': '4bb9ece8-916f-4c4b-9526-16b7e7cff583', 'status': 'PROCESSING', 'overall_progress_percentage': 79, 'total_jobs': 10, 'completed_jobs': 3, 'failed_jobs': 0, 'jobs': [{'url': 'https://en.wikipedia.org/wiki/Mathematics', 'doc_id': '290db3c3-e12f-490a-afeb-a7e8c5855e68', 'job_id': '2e630a53-e1d2-463b-aef3-66891bbb6903', 'status': 'EMBEDDING', 'progress_percentage': 70, 'error_message': None}, {'url': 'https://en.wikipedia.org/wiki/Number_theory', 'doc_id': '97683bbf-e337-4122-b321-e7b28795e2d5', 'job_id': '94b21d2f-9743-448d-88dd-dbe13754dd4a', 'status': 'EMBEDDING', 'progress_percentage': 70, 'error_message': None}, {'url': 'https://en.wikipedia.org/wiki/Calculus', 'doc_id': '475a72df-ba15-4e83-a295-fb7ee3faf5cc', 'job_id': '603e0d6e-019f-462e-b3ed-2cabfbfbf880', 'status': 'EMBEDDING', 'progress_percentage': 70, 'error_message': None}, {'url': 'https://en.wikipedia.org/wiki/Linear_algebra',